# 🤖 J-Quants Interactive Analyst

セッションベースの対話型分析環境です。
- `Session` = 1つの分析テーマ（スイング、マクロなど）
- 会話・計算結果は全てディスクに永続化されます
- カーネル再起動後も `Session("同じ名前")` で完全復元されます

In [2]:
import sys
from pathlib import Path

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from dotenv import load_dotenv
load_dotenv(project_root / ".env")

# マジックコマンド %%chat を有効化
%load_ext src.agent.magic

## セッションの作成（または再開）

新規の場合は `data_range` を指定します。再開する場合は名前だけでOKです。

In [4]:
from src.agent.session import Session

# 新規セッション（データを読み込む）
session = Session(
    name="swing_2026518",
    data_dir=str(project_root / "data"),
)

# 翌日の再開時はこれだけ:
# session = Session("swing_20240515", data_dir=str(project_root / "data"))

print(f"Session: {session.name}")
# ★ session.df が None の場合でもエラーにならないように修正！
print(f"Data: {len(session.df) if session.df is not None else 'None (AI Dynamic Mode)'} rows")
print(f"Artifacts: {session.list_artifacts()}")


INFO:src.agent.core:Using model: gemini-2.5-flash


Session: swing_2026518
Data: None (AI Dynamic Mode) rows
Artifacts: []


## AIとの対話

`%%chat` で日本語の指示を送るだけです。AIはデータの状態を常に把握しています。

In [5]:
%%chat
私はスイングトレード用の面白い投資戦略を求めています。
あなたが「最も適したデータ期間」を自分で決定してデータをロードし、
そのデータに対してシミュレーションを行って、面白い結果が出た銘柄とその考察を教えてください。

*(**swing_2026518** が考え中...)*

INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
ERROR:src.analyzer.runner:Error executing AI code: 'Code'


**🤖 swing_2026518**:

伝説のクオンツ・トレーダーである私が、スイングトレード用の「底打ち反発狙い」戦略を考案し、過去のデータでシミュレーションを行いました。

### 1. 仮説立案
現在の市場環境では、一時的な調整や押し目で下落したものの、中長期的な上昇トレンドを維持している銘柄が、出来高を伴って反発する際に、短期的な買いのチャンスとなることが多いと見ています。特に、市場の注目が集まり出来高が急増するポイントは、反発の確度を高める重要なシグナルです。

### 2. 戦略「底打ち反発狙い」
以下の条件でエントリーし、5営業日後に売却するシミュレーションを行います。

-   **データ期間**: 2025年5月1日から2026年5月17日まで（約1年間）
-   **エントリー条件（当日）**:
    1.  **トレンド**: 25日移動平均線が75日移動平均線を上回っている（中長期的な上昇トレンド）。
    2.  **下落**: 当日の終値が、過去5営業日の最高値から10%以上下落している（一時的な調整や押し目）。
    3.  **反発の兆し**: 当日の終値が前日の終値より高い。
    4.  **出来高増加**: 当日の調整済み出来高が過去20日間の平均調整済み出来高の1.5倍以上（買い圧力の増加）。
-   **エグジット条件**:
    *   エントリーシグナル発生日の翌営業日の寄り付きで買い、その5営業日後の終値で売却します。

このシミュレーションで、最低5回以上シグナルが発生し、かつ平均利益率が高かった銘柄を抽出します。

### 3. 分析コード

```python
# 最も適したデータ期間として、過去1年強のデータをロード
df = runner.load_daily_bars("2025-05-01", "2026-05-17")

# 銘柄ごとにグループ化して計算
df_grouped = df.groupby('Code')

# 1. 移動平均線と出来高移動平均の計算
df['AdjC_MA25'] = df_grouped['AdjC'].transform(lambda x: x.rolling(window=25).mean())
df['AdjC_MA75'] = df_grouped['AdjC'].transform(lambda x: x.rolling(window=75).mean())
df['AdjVo_MA20'] = df_grouped['AdjVo'].transform(lambda x: x.rolling(window=20).mean())

# 2. 過去5営業日の最高終値と前日終値の計算
# shift(1)で当日のデータを含まない過去の最高値を取得
df['AdjC_High5D'] = df_grouped['AdjC'].transform(lambda x: x.rolling(window=5).max().shift(1))
df['AdjC_Prev'] = df_grouped['AdjC'].transform(lambda x: x.shift(1))

# 計算に必要なカラムのNaNを除外
df = df.dropna(subset=['AdjC_MA75', 'AdjVo_MA20', 'AdjC_High5D', 'AdjC_Prev'])

# 3. エントリー条件の定義
# 条件1: 25日移動平均線が75日移動平均線を上回っている
condition1 = (df['AdjC_MA25'] > df['AdjC_MA75'])
# 条件2: 当日の終値が、過去5営業日の最高値から10%以上下落している
condition2 = (df['AdjC'] / df['AdjC_High5D'] - 1 < -0.10)
# 条件3: 当日の終値が前日の終値より高い（反発の兆し）
condition3 = (df['AdjC'] > df['AdjC_Prev'])
# 条件4: 当日の調整済み出来高が過去20日間の平均調整済み出来高の1.5倍以上
condition4 = (df['AdjVo'] > df['AdjVo_MA20'] * 1.5)

# 全ての条件を満たす日をエントリーシグナルとする
df['entry_signal'] = condition1 & condition2 & condition3 & condition4

# エントリーシグナルが出た日の次の営業日の寄り付きと、その5営業日後の終値を計算
# shift(-1) は次の行、shift(-5) は5行後のデータを取得。groupby('Code')内で実行することで、銘柄ごとの連続性を保つ。
df['buy_price'] = df_grouped['AdjO'].transform(lambda x: x.shift(-1))
df['sell_price'] = df_grouped['AdjC'].transform(lambda x: x.shift(-5))

# シグナルが出た日のみに絞り込み
signals = df[df['entry_signal']].copy()

# 利益率の計算 (NaNがある場合は除外)
signals['profit_rate'] = (signals['sell_price'] / signals['buy_price'] - 1)
signals = signals.dropna(subset=['profit_rate'])

# 各銘柄の平均利益率とシグナル発生回数を計算
# シグナル発生回数が少ない銘柄は信頼性が低いので、ある程度の回数があった銘柄に絞る
min_signals = 5 # 最低5回はシグナルが出た銘柄を対象とする

results = signals.groupby('Code').agg(
    average_profit_rate=('profit_rate', 'mean'),
    num_signals=('entry_signal', 'sum')
).reset_index()

# 最低シグナル回数でフィルタリングし、平均利益率でソート
output_df = results[results['num_signals'] >= min_signals].sort_values(by='average_profit_rate', ascending=False)

# 上位10銘柄に絞る
output_df = output_df.head(10)

# 考察の作成
if not output_df.empty:
    result = "過去1年間（2025-05-01から2026-05-17）のデータを用いて、スイングトレード戦略「底打ち反発狙い」のシミュレーションを行いました。\n"
    result += "この戦略は、上昇トレンド中の銘柄が一時的に大きく下落し、かつ出来高を伴って反発の兆しを見せた際に、翌営業日の寄り付きで買い、5営業日後に終値で売却するというものです。\n"
    result += f"最低{min_signals}回以上のシグナルが発生した銘柄の中から、平均利益率が高かった上位10銘柄を抽出しました。\n\n"
    result += "これらの銘柄は、過去の市場環境において、短期的な調整後に強い反発力を見せる傾向があったと考えられます。特に、出来高の急増は市場の関心の高まりを示唆しており、反発の確度を高める要因となった可能性があります。\n"
    result += "今後のトレードでは、これらの銘柄が同様の条件を満たした際に、買いを検討する価値があるかもしれません。ただし、過去のパフォーマンスは将来の結果を保証するものではないため、現在の市場環境や個別銘柄のファンダメンタルズも考慮に入れることが重要です。\n"
else:
    result = "指定された期間と戦略では、シミュレーションの結果、条件を満たす銘柄が見つかりませんでした。戦略の条件を緩和するか、データ期間を調整することをご検討ください。"
```

---

❌ **実行エラー**

```text
'Code'
```

## 深掘り（AIは前回の結果を知っている）

In [4]:
%%chat
その結果の中から、さらに終値が2000円以下の銘柄だけに絞り込んで。

*(**swing_20240515** が考え中...)*

INFO:google_genai.models:AFC is enabled with max remote calls: 10.
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
INFO:src.agent.session:Saved artifact: step_002.parquet (43 rows)


**🤖 swing_20240515**:

```python
# 前回の結果 (output_df) から、終値が2000円以下の銘柄をフィルタリング
condition = output_df['AdjC'] <= 2000
output_df = output_df[condition]

# 結果の考察
if not output_df.empty:
    result = f"終値が始値より5%以上高く、かつ終値が2000円以下だった銘柄は {len(output_df)} 件ありました。\n" \
             f"特に出来高が多かった上位5銘柄は以下の通りです:\n" \
             f"{output_df[['Date', 'Code', 'AdjO', 'AdjC', 'AdjVo']].head().to_string(index=False)}"
else:
    result = "終値が始値より5%以上高く、かつ終値が2000円以下の銘柄は見つかりませんでした。"
```

---

✅ **実行完了**

> 終値が始値より5%以上高く、かつ終値が2000円以下だった銘柄は 43 件ありました。
特に出来高が多かった上位5銘柄は以下の通りです:
      Date  Code   AdjO   AdjC      AdjVo
2024-05-15 65740   12.0   13.8 84420000.0
2024-05-15 67400   18.0   19.0 22318800.0
2024-05-15 52160  252.0  328.0 16841600.0
2024-05-15 77710   95.0  103.0 11492400.0
2024-05-15 15180 1078.0 1136.0  8897500.0

,Date,Code,O,H,L,C,UL,LL,Vo,Va,AdjFactor,AdjO,AdjH,AdjL,AdjC,AdjVo
2740,2024-05-15,65740,1204.0,1384.0,1113.0,1384.0,1,0,844200.0,1.066995e+09,1.0,12.0,13.8,11.1,13.8,84420000.0
2823,2024-05-15,67400,18.0,19.0,18.0,19.0,0,0,22318800.0,4.036973e+08,1.0,18.0,19.0,18.0,19.0,22318800.0
2102,2024-05-15,52160,252.0,332.0,248.0,328.0,1,0,16841600.0,5.105079e+09,1.0,252.0,332.0,248.0,328.0,16841600.0
3405,2024-05-15,77710,95.0,107.0,92.0,103.0,1,0,11492400.0,1.183465e+09,1.0,95.0,107.0,92.0,103.0,11492400.0
128,2024-05-15,15180,5390.0,6210.0,5390.0,5680.0,0,0,1779500.0,1.020472e+10,1.0,1078.0,1242.0,1078.0,1136.0,8897500.0
3996,2024-05-15,92270,1200.0,1320.0,1200.0,1290.0,0,0,2924300.0,3.738118e+09,1.0,1200.0,1320.0,1200.0,1290.0,2924300.0
1813,2024-05-15,45640,15.0,16.0,15.0,16.0,0,0,2861900.0,4.344950e+07,1.0,15.0,16.0,15.0,16.0,2861900.0
979,2024-05-15,31030,215.0,234.0,213.0,231.0,0,0,2635100.0,5.994758e+08,1.0,215.0,234.0,213.0,231.0,2635100.0
3714,2024-05-15,83460,338.0,370.0,338.0,359.0,0,0,2466500.0,8.795353e+08,1.0,338.0,370.0,338.0,359.0,2466500.0
689,2024-05-15,25850,5500.0,5900.0,5370.0,5790.0,0,0,510400.0,2.883771e+09,1.0,1375.0,1475.0,1342.5,1447.5,2041600.0


## 自分のコードで計算結果を操作する

In [5]:
# AIが計算した最新の output_df を自分で操作できる
if session.latest_output_df is not None:
    print(session.latest_output_df.describe())
else:
    print("まだ計算結果はありません。")

                      Date            O            H            L  \
count                   43    43.000000    43.000000    43.000000   
mean   2024-05-15 00:00:00  1181.697674  1299.453488  1163.269767   
min    2024-05-15 00:00:00    15.000000    16.000000    15.000000   
25%    2024-05-15 00:00:00   323.000000   354.500000   322.500000   
50%    2024-05-15 00:00:00   926.000000  1054.000000   926.000000   
75%    2024-05-15 00:00:00  1427.000000  1601.500000  1363.500000   
max    2024-05-15 00:00:00  5500.000000  6210.000000  5390.000000   
std                    NaN  1288.311825  1418.451532  1272.739234   

                 C            Vo            Va  AdjFactor         AdjO  \
count    43.000000  4.300000e+01  4.300000e+01       43.0    43.000000   
mean   1270.209302  1.885377e+06  9.303498e+08        1.0   770.565116   
min      16.000000  2.400000e+03  1.160000e+06        1.0    12.000000   
25%     345.000000  1.347500e+05  4.415975e+07        1.0   338.550000   
50%    1

## 保存された計算結果（artifacts）の確認

In [ ]:
# 各ステップの結果はParquetとして永続化されている
print("Saved artifacts:", session.list_artifacts())

# 過去のステップの結果を呼び出すこともできる
# df_step1 = session.load_artifact(1)